# 01 - Plain text NER (Task 1)
Run GLiNER over `data/plain/validation.jsonl`, save predictions and evaluate them
with strict and relaxed matching.

In [6]:
import os, sys, warnings
from pathlib import Path

os.environ["HF_HUB_DISABLE_XET"] = "1"                 # evitar la descarga que se colgaba
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"    # ocultar aviso de symlinks
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

sys.path.insert(0, str(Path('..').resolve()))          # para poder importar 'ner'

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
from ner.evaluation import evaluate_ner_strict, evaluate_ner_relaxed, ENTITY_LABELS
from ner.inference import load_gliner_model, split_into_chunks, predict_dataset
from ner.utils import load_jsonl, save_responses_to_jsonl

In [8]:
VALIDATION_PATH = '../data/plain/validation.jsonl'
OUTPUT_PATH = '../results/plain/gliner_small.jsonl'     # nuestras predicciones

LABELS = list(ENTITY_LABELS)                            # las 7 etiquetas del proyecto

In [9]:
samples = load_jsonl(VALIDATION_PATH)
doc_text = samples[0]['text']
chunks = split_into_chunks(doc_text)

print('Document length:', len(doc_text))
print('Number of chunks:', len(chunks))
for offset, chunk in chunks:
    ok = doc_text[offset:offset + len(chunk)] == chunk     # ¿el offset es correcto?
    print(f'offset={offset:<5} length={len(chunk):<5} correct={ok}')

Document length: 4091
Number of chunks: 5
offset=0     length=927   correct=True
offset=927   length=904   correct=True
offset=1831  length=834   correct=True
offset=2665  length=848   correct=True
offset=3513  length=578   correct=True


## Baseline: GLiNER small, threshold 0.5, no label descriptions

In [5]:
model = load_gliner_model()        # sin device -> elige CPU o GPU solo
responses = predict_dataset(model, samples, LABELS, threshold=0.5)
save_responses_to_jsonl(responses, OUTPUT_PATH)
print('Saved', len(responses), 'documents in', OUTPUT_PATH)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

GLiNER:   0%|          | 0/50 [00:00<?, ?it/s]

Saved 50 documents in ../results/plain/gliner_small.jsonl


In [10]:
strict = evaluate_ner_strict(VALIDATION_PATH, OUTPUT_PATH)
relaxed = evaluate_ner_relaxed(VALIDATION_PATH, OUTPUT_PATH)


STRICT EVALUATION
Micro -> P: 0.74 | R: 0.85 | F1: 0.79 | TP: 1743 | FP: 610 | FN: 307

ENTITY TYPE                    P      R     F1      TP      FP      FN
--------------------------------------------------------------------------------------------
PERSON_NAME                 0.96   1.00   0.98     500      23       0
ID_CARD_NUMBER              0.82   0.81   0.82     163      36      37
ADDRESS                     0.03   0.06   0.04      11     365     189
REGISTRATION_NUMBER         0.93   0.93   0.93     232      17      18
DATE                        0.99   0.99   0.99     398       2       2
AMOUNT                      0.74   0.84   0.79     169      58      31
ORGANIZATION                0.71   0.90   0.80     270     109      30

RELAXED EVALUATION - OCCURRENCES
Micro -> P: 0.74 | R: 0.85 | F1: 0.79 | TP: 1743 | FP: 610 | FN: 307

ENTITY TYPE                    P      R     F1      TP      FP      FN
---------------------------------------------------------------------------

In [11]:
LABEL_TO_CHECK = 'ADDRESS'      # luego cámbialo a 'ORGANIZATION'

print('GOLD:')
for e in samples[0]['entities']:
    if e['label'] == LABEL_TO_CHECK:
        print('   ', repr(e['text']))

print('\nPREDICTED:')
for e in responses[0]['entities']:
    if e['label'] == LABEL_TO_CHECK:
        print('   ', repr(e['text']), f"({e['score']:.2f})")

GOLD:
    'Avenida de Aragón, 102, 2-B, 17001 Girona'
    'Calle de la Estación, 38, 6-2a, 08221 Terrassa'
    'Avenida de Aragón, 102, 2-B, 17001 Girona'
    'Calle de la Estación, 38, 6-2a, 08221 Terrassa'

PREDICTED:
    'Avenida de Aragón' (0.91)
    '17001 Girona' (0.86)
    'Calle de la Estación' (0.80)
    '08221 Terrassa' (0.76)
    'Avenida de Aragón' (0.93)
    '102, 2-B' (0.65)
    '17001 Girona' (0.88)
    'Calle de la Estación, 38, 6-2a, 08221 Terrassa' (0.95)


### Baseline analysis (GLiNER small, threshold 0.5)

- **Strict micro:** P 0.74 | R 0.85 | F1 0.79 (teacher's reference: F1 0.81).
- **Best types:** DATE (0.99), PERSON_NAME (0.98) and REGISTRATION_NUMBER (0.93).
- **Worst type: ADDRESS (F1 0.04).** The model finds the addresses but splits them into pieces:
  the gold *Avenida de Aragón, 102, 2-B, 17001 Girona* is predicted as *Avenida de Aragón*,
  *102, 2-B* and *17001 Girona*. Each piece is a false positive and the full address a false negative.
- **ORGANIZATION (0.80)** has the same problem with long names such as
  *Servicio de Procedimientos Públicos de la Región Central*.
- **ID_CARD_NUMBER:** recall is 0.98 in the unique view but lower in occurrences, so the model
  finds every ID but misses some repeated mentions.
- **Strict = relaxed occurrences:** in plain text the predicted text is the same as the gold
  text, so the errors are not caused by shifted spans but by split or wrong entities.

### Improvement 1: merge adjacent pieces of ADDRESS and ORGANIZATION
GLiNER splits long entities. After prediction, consecutive entities with the same label are merged
when the text between them is only commas/numbers (ADDRESS) or a preposition (ORGANIZATION).

In [12]:
from ner.inference import merge_dataset

MERGED_PATH = '../results/plain/gliner_small_merge.jsonl'

# --- Cargar las predicciones de la baseline y unir los trozos ---
responses = load_jsonl(OUTPUT_PATH)
merged = merge_dataset(responses)
save_responses_to_jsonl(merged, MERGED_PATH)

# --- Evaluar ---
strict_merge = evaluate_ner_strict(VALIDATION_PATH, MERGED_PATH)
relaxed_merge = evaluate_ner_relaxed(VALIDATION_PATH, MERGED_PATH)


STRICT EVALUATION
Micro -> P: 0.88 | R: 0.92 | F1: 0.90 | TP: 1894 | FP: 270 | FN: 156

ENTITY TYPE                    P      R     F1      TP      FP      FN
--------------------------------------------------------------------------------------------
PERSON_NAME                 0.96   1.00   0.98     500      23       0
ID_CARD_NUMBER              0.82   0.81   0.82     163      36      37
ADDRESS                     0.70   0.72   0.71     144      62      56
REGISTRATION_NUMBER         0.93   0.93   0.93     232      17      18
DATE                        0.99   0.99   0.99     398       2       2
AMOUNT                      0.74   0.84   0.79     169      58      31
ORGANIZATION                0.80   0.96   0.87     288      72      12

RELAXED EVALUATION - OCCURRENCES
Micro -> P: 0.88 | R: 0.92 | F1: 0.90 | TP: 1894 | FP: 270 | FN: 156

ENTITY TYPE                    P      R     F1      TP      FP      FN
---------------------------------------------------------------------------

In [13]:
for label in ['ADDRESS', 'ORGANIZATION']:
    print(f'--- {label} (after merge) ---')
    for e in merged[0]['entities']:
        if e['label'] == label:
            print('   ', repr(e['text']))

--- ADDRESS (after merge) ---
    'Avenida de Aragón, 102, 2-B, 17001 Girona'
    'Calle de la Estación, 38, 6-2a, 08221 Terrassa'
    'Avenida de Aragón, 102, 2-B, 17001 Girona'
    'Calle de la Estación, 38, 6-2a, 08221 Terrassa'
--- ORGANIZATION (after merge) ---
    'Servicio de Procedimientos Públicos de la Región Central'
    'Instituto de Coordinación Territorial del Área Metropolitana'
    'Unidad de Atención Ciudadana de la Comarca Interior'
    'Dirección de Supervisión Administrativa de Servicios Intermunicipales'
    'Unidad de Atención Ciudadana de la Comarca Interior'
    'Servicio de Procedimientos Públicos de la Región Central'


**Result of the merge post-processing:**
- Strict micro F1 improves from **0.79 to 0.90** (P 0.74 → 0.88, R 0.85 → 0.92).
- ADDRESS goes from F1 0.04 to **0.71** and ORGANIZATION from 0.80 to **0.87**.
  The other labels do not change, because the rule only applies to these two.
- Example: *Avenida de Aragón* + *102, 2-B* + *17001 Girona* → *Avenida de Aragón, 102, 2-B, 17001 Girona*.
- ADDRESS still has errors (62 FP, 56 FN): sometimes a piece is missing or labelled with another
  type, so the merge cannot rebuild the full address.

In [14]:
RESULTS = []   # una fila por experimento y tipo de evaluación

def add_result(input_name, model_name, descriptions, extraction, predictions_path):
    """Evaluate a predictions file and store micro P/R/F1 for the 3 evaluation views."""
    strict = evaluate_ner_strict(VALIDATION_PATH, predictions_path, print_results=False)
    relaxed = evaluate_ner_relaxed(VALIDATION_PATH, predictions_path, print_results=False)
    views = {'Strict': strict,
             'Relaxed occurrences': relaxed['occurrences'],
             'Relaxed unique': relaxed['unique']}
    for evaluation, metrics in views.items():
        m = metrics['micro']
        RESULTS.append([input_name, model_name, descriptions, extraction, evaluation,
                        f"{m['precision']:.2f}", f"{m['recall']:.2f}", f"{m['f1']:.2f}"])

def show_results():
    """Print RESULTS as a Markdown table (ready to paste in the report)."""
    header = ['Input', 'Model', 'Label descriptions', 'Extraction', 'Evaluation', 'P', 'R', 'F1']
    print('| ' + ' | '.join(header) + ' |')
    print('|' + '---|' * len(header))
    for row in RESULTS:
        print('| ' + ' | '.join(row) + ' |')

# --- Los dos experimentos que ya tenemos ---
add_result('Plain text', 'gliner_small', 'No', '—', OUTPUT_PATH)
add_result('Plain text', 'gliner_small + merge', 'No', '—', MERGED_PATH)
show_results()

| Input | Model | Label descriptions | Extraction | Evaluation | P | R | F1 |
|---|---|---|---|---|---|---|---|
| Plain text | gliner_small | No | — | Strict | 0.74 | 0.85 | 0.79 |
| Plain text | gliner_small | No | — | Relaxed occurrences | 0.74 | 0.85 | 0.79 |
| Plain text | gliner_small | No | — | Relaxed unique | 0.72 | 0.88 | 0.79 |
| Plain text | gliner_small + merge | No | — | Strict | 0.88 | 0.92 | 0.90 |
| Plain text | gliner_small + merge | No | — | Relaxed occurrences | 0.88 | 0.92 | 0.90 |
| Plain text | gliner_small + merge | No | — | Relaxed unique | 0.84 | 0.94 | 0.89 |


### Improvement 2: label descriptions
Instead of the label codes, the model receives a natural language description of each entity type.
The predicted descriptions are mapped back to the original labels before the evaluation.

In [15]:
from ner.inference import predict_dataset, merge_dataset

# --- Descripción en lenguaje natural de cada etiqueta ---
LABEL_DESCRIPTIONS = {
    'PERSON_NAME': 'person full name',
    'ID_CARD_NUMBER': 'national identity document number (DNI or NIE)',
    'ADDRESS': 'full postal address with street, number, floor, postal code and city',
    'REGISTRATION_NUMBER': 'administrative file or registration number',
    'DATE': 'date',
    'AMOUNT': 'amount of money',
    'ORGANIZATION': 'public organization, institution or administrative office',
}

DESC_PATH = '../results/plain/gliner_small_desc_merge.jsonl'

samples = load_jsonl(VALIDATION_PATH)
model = load_gliner_model()     # carga desde el disco (unos segundos)

# --- Predecir con descripciones, unir trozos y guardar ---
responses_desc = predict_dataset(model, samples, LABELS, threshold=0.5,
                                 label_descriptions=LABEL_DESCRIPTIONS)
save_responses_to_jsonl(merge_dataset(responses_desc), DESC_PATH)

# --- Añadir a la tabla comparativa ---
add_result('Plain text', 'gliner_small + merge', 'Yes', '—', DESC_PATH)
show_results()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

GLiNER:   0%|          | 0/50 [00:00<?, ?it/s]

| Input | Model | Label descriptions | Extraction | Evaluation | P | R | F1 |
|---|---|---|---|---|---|---|---|
| Plain text | gliner_small | No | — | Strict | 0.74 | 0.85 | 0.79 |
| Plain text | gliner_small | No | — | Relaxed occurrences | 0.74 | 0.85 | 0.79 |
| Plain text | gliner_small | No | — | Relaxed unique | 0.72 | 0.88 | 0.79 |
| Plain text | gliner_small + merge | No | — | Strict | 0.88 | 0.92 | 0.90 |
| Plain text | gliner_small + merge | No | — | Relaxed occurrences | 0.88 | 0.92 | 0.90 |
| Plain text | gliner_small + merge | No | — | Relaxed unique | 0.84 | 0.94 | 0.89 |
| Plain text | gliner_small + merge | Yes | — | Strict | 0.84 | 0.83 | 0.83 |
| Plain text | gliner_small + merge | Yes | — | Relaxed occurrences | 0.84 | 0.83 | 0.83 |
| Plain text | gliner_small + merge | Yes | — | Relaxed unique | 0.81 | 0.89 | 0.85 |


**Result of label descriptions:** they make the model **worse** (strict F1 0.90 → 0.83).
The biggest drop is in recall (0.92 → 0.83): with long descriptions the model misses entities that
it found with the short labels. A possible reason is that GLiNER was mainly trained with short
entity type names, and our label names were already quite descriptive.
**Decision:** we keep the short labels (no descriptions) + merge for the next experiments.

## Task 1 — Conclusions
| Configuration | Strict F1 |
|---|---|
| gliner_small (baseline) | 0.79 |
| gliner_small + merge | **0.90** |
| gliner_small + merge + label descriptions | 0.83 |

- The main problem of the baseline was that GLiNER splits long entities (ADDRESS, ORGANIZATION).
  Merging adjacent pieces with the same label improves strict F1 from 0.79 to 0.90.
- Label descriptions made the results worse, so we keep the short labels.
- **Configuration used for the PDF tasks:** gliner_small, threshold 0.5, short labels, merge.